# Training Sentiment Analysis: IndoBERT + BiLSTM (3-class)

Notebook ini melatih model **IndoBERT + BiLSTM** untuk klasifikasi sentimen berita (Negative/Neutral/Positive).

**Input**: tiga dataframe seed berlabel manual: `df1` (CNBC), `df2` (Detik), `df3` (Kompas). Masing-masing minimal memiliki kolom: `date, title, content, article_id, text, label`.

**Catatan**: HuggingFace `Trainer` mengharuskan label berupa `0..C-1`, sehingga label `-1/0/1` akan di-map ke `0/1/2`.


**Notes** : 
1. Batch size di notebook BiLSTM saya set default 8 (lebih aman untuk RAM/GPU). Jika GPU kuat, bisa naik ke 16.
2. MAX_LENGTH = 256 tetap rasional berdasarkan EDA Anda.
3. Saya sediakan flag:
    - FREEZE_BERT = False (default)
    - jika Anda ingin baseline cepat, bisa set True (tapi performa biasanya turun)

**Checklist komparasi yang “fair” untuk Bab 4**

Agar komparasi IndoBERT vs IndoBERT+BiLSTM valid:

split train/val sama (SEED=42, stratified)

max_length sama

epochs sama (mis. 3)

metrik utama: macro-F1

In [ ]:

# =========================
# Imports
# =========================
import os
import random
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

from transformers import (
    AutoTokenizer,
    AutoModel,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed
)
from transformers.modeling_outputs import SequenceClassifierOutput


In [ ]:

SEED = 42
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:

# =========================
# Combine seed dataframes
# =========================
# Pastikan df1, df2, df3 sudah ada di memory sebelum menjalankan cell ini.
required_cols = ["date", "title", "content", "article_id", "text", "label"]

def _select_cols(df: pd.DataFrame) -> pd.DataFrame:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    return df[required_cols].copy()

df_seed = pd.concat([_select_cols(df1), _select_cols(df2), _select_cols(df3)], ignore_index=True)

print("seed shape:", df_seed.shape)
df_seed.head()


In [ ]:

# =========================
# Basic validation
# =========================
print(df_seed.dtypes)
print("\nNull counts:\n", df_seed.isna().sum())

# label harus -1/0/1
assert set(df_seed["label"].unique()).issubset({-1, 0, 1}), f"Unexpected labels: {sorted(df_seed['label'].unique().tolist())}"


In [ ]:

# =========================
# Quick EDA
# =========================
label_name = df_seed["label"].map({-1:"Negative", 0:"Neutral", 1:"Positive"})
print("Label distribution:\n", label_name.value_counts())

df_seed["text_len_words"] = df_seed["text"].astype(str).str.split().apply(len)
print("\nText length (words) summary:\n", df_seed["text_len_words"].describe())


In [ ]:

# =========================
# Train/Validation split (stratified)
# =========================
train_df, val_df = train_test_split(
    df_seed,
    test_size=0.2,
    stratify=df_seed["label"],
    random_state=SEED
)

print("Train:", train_df.shape, "Val:", val_df.shape)


In [ ]:

# =========================
# Remap labels: -1/0/1 -> 0/1/2
# =========================
label_to_id = {-1: 0, 0: 1, 1: 2}
id_to_label = {0: "negative", 1: "neutral", 2: "positive"}

train_df = train_df.copy()
val_df = val_df.copy()
train_df["label_id"] = train_df["label"].map(label_to_id)
val_df["label_id"] = val_df["label"].map(label_to_id)

assert train_df["label_id"].notna().all()
assert val_df["label_id"].notna().all()


In [ ]:

# =========================
# Tokenizer
# =========================
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


In [ ]:

class NewsDataset(Dataset):
    def __init__(self, texts: pd.Series, labels: pd.Series, tokenizer, max_length: int = 256):
        self.texts = texts.astype(str).tolist()
        self.labels = labels.astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding=False,
            max_length=self.max_length
        )
        enc["labels"] = self.labels[idx]
        return enc


In [ ]:

train_dataset = NewsDataset(train_df["text"], train_df["label_id"], tokenizer, MAX_LENGTH)
val_dataset = NewsDataset(val_df["text"], val_df["label_id"], tokenizer, MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:

class IndoBertBiLSTM(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_labels: int = 3,
        lstm_hidden: int = 256,
        lstm_layers: int = 1,
        dropout: float = 0.2,
        freeze_bert: bool = False,
        id_to_label: dict | None = None,
        label_to_id: dict | None = None,
    ):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        if freeze_bert:
            for p in self.bert.parameters():
                p.requires_grad = False

        self.lstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden * 2, num_labels)
        self.num_labels = num_labels

        # optional label mapping for nicer logs
        if id_to_label is not None:
            self.config = type("cfg", (), {})()
            self.config.id2label = id_to_label
            self.config.label2id = label_to_id if label_to_id is not None else None

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        x = bert_out.last_hidden_state  # (B, T, H)

        # lengths for packing (avoid zeros)
        lengths = attention_mask.sum(dim=1).clamp(min=1).cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed)

        # h_n shape: (num_layers*2, B, hidden)
        # take last layer's forward & backward
        if h_n.size(0) >= 2:
            h_fwd = h_n[-2]
            h_bwd = h_n[-1]
            h = torch.cat([h_fwd, h_bwd], dim=1)  # (B, hidden*2)
        else:
            h = h_n[-1]

        h = self.dropout(h)
        logits = self.classifier(h)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        return SequenceClassifierOutput(loss=loss, logits=logits)


In [ ]:

FREEZE_BERT = False  # set True untuk baseline cepat, tapi biasanya performa lebih rendah

model = IndoBertBiLSTM(
    model_name=MODEL_NAME,
    num_labels=3,
    lstm_hidden=256,
    lstm_layers=1,
    dropout=0.2,
    freeze_bert=FREEZE_BERT,
    id_to_label=id_to_label,
    label_to_id={v:k for k,v in id_to_label.items()},
).to(device)

sum(p.numel() for p in model.parameters() if p.requires_grad), "trainable parameters"


In [ ]:

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }


In [ ]:

training_args = TrainingArguments(
    output_dir="./indobert_bilstm_sentiment",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,   # BiLSTM menambah memory, mulai dari 8
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=2,
    report_to="none",
)


In [ ]:

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


In [ ]:

trainer.train()


In [ ]:

pred = trainer.predict(val_dataset)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=1)

print(classification_report(y_true, y_pred, target_names=["negative","neutral","positive"]))


In [ ]:

trainer.save_model("indobert_bilstm_sentiment_news")
tokenizer.save_pretrained("indobert_bilstm_sentiment_news")
